In [19]:
%pwd

'd:\\saad\\FILE LOG\\RAG'

In [20]:
import os
os.chdir('../')
%pwd

'd:\\saad\\FILE LOG'

RAG Pipelines Data Ingestion to vector DB Pipeline

In [21]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_community.document_loaders import DirectoryLoader
from  pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [22]:
### read all Pdf inside directory
def process_all_pdf(pdf_directory):   
    all_document = []
    pdf_dir = Path(pdf_directory)
    
    pdf_files = list(pdf_dir.glob('**/*.pdf'))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"Processing:{pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_document.extend(documents)
            print(f'loaded {len(documents)} page')
        
        except Exception as e:
            print(f"X error: {e}")
    
    print(f"Total documents loaded:{len(all_document)}")
    return all_document      

all_pdf_documents = process_all_pdf('data')           
                

Found 0 PDF files to process
Total documents loaded:0


In [23]:
all_pdf_documents

[]

In [24]:
### text splition in chunk

def spliting_document(documents,chunk_size=1000,chunk_overlap=200):
    """Split document into smaller chunk for batter RAG performence"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_doc = text_splitter.split_documents(documents)
    print(f"split {len(documents)} document into {len(split_doc)} chunks")
    
    
    if split_doc:
        print(f"\nExample of chunk")
        print(f"Content: {split_doc[0].page_content[:200]}....")
        print(f"MetaData: {split_doc[0].metadata}")
    
    return split_doc    
    

In [25]:
chunks=spliting_document(all_pdf_documents)


split 0 document into 0 chunks


In [26]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [27]:
class embadding_manager:
    """Handles document embadding generation using SentanceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-V2"):
        """
        Initialize the embading manager
        
        Args:
             model_name: Huggingface model name for sentence embadings 

        """
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """Load sentenceTransformer Model"""
        try:
            print(f"Loding embadding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)    
            print(f"Model loaded sucessfully. Embadding dimension: {self.model.get_embedding_dimension()}")  ##
        
        except Exception as e:
            print(f"Error loading model { self.model_name}: {e}")
            raise    
        
    def generate_embaddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text string to embed
            
        returns:
             numpy array of embadings with shape (len(texts), embading_dim)    
         
        """        
        if not self.model:
            raise ValueError("model not loaded")
        
        print(f"Generating embaddings for {len(texts)} texts...")
        embaddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embadings with shape: {embaddings.shape}")
        return embaddings
    
    # def get_embading_dimension(self) -> int:
    #     """Get the embadding dimension of the model"""
    #     if not self.model:
    #         raise ValueError("Model not Loaded")
        
    #     return self.model.get_embedding_dimension()       ##
    

Manager = embadding_manager()
Manager      
        

Loding embadding model: all-MiniLM-L6-V2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded sucessfully. Embadding dimension: 384


In [28]:
class VectorSrore:
    """Manages document embaddings in a ChromaDB vector store"""
    
    def __init__(self,collection_name: str = "pdf_documents",persist_directory: str= "data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
        
    def _initialize_store(self):
        """Initialize ChoroaB=DB client and collection""" 
        try:   
            #Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory) 
            
            #Get or Create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embadings for RAG"}
            )
            print(f"Vectore store initialize. collection: {self.collection_name}")
            print(f"Existing document in collection: {self.collection.count()}")
        
        except Exception as e:
              print(f"Error initializing vector store: {e}") 
              raise 
          
    def add_pdf_documents(self, documents: List[Any], embadings: np.ndarray):
        """ 
        Add documents and their embadings to the vector store
        
        Args:
            documents: List of Langchain documents
            embadings: Corresponding embading for the documents
        """     
        if len(documents) != len(embadings):
            raise ValueError("Number of documents must match number of embadings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        ids = []
        metadatas = []
        documents_text = []
        embading_list = []
        
        for i, (doc, embading) in enumerate(zip(documents, embadings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            #prepare metadeta
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content) 
            metadatas.append(metadata)
            
            #Document content
            documents_text.append(doc.page_content)
            
            #Embading
            embading_list.append(embading.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embading_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(f"sucessfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection {self.collection.count()}")
        
        except Exception as e:
               print(f"error adding documents to vector store; {e}")
               raise
           
           
vectorstore=VectorSrore()
vectorstore            

Vectore store initialize. collection: pdf_documents
Existing document in collection: 488


In [29]:
chunks

[]

In [30]:
### convert the text to embadings
texts=[doc.page_content for doc in chunks]

##genrate the embading
embading=Manager.generate_embaddings(texts)

###Store in vectore database
vectorstore.add_pdf_documents(chunks,embading)

Generating embaddings for 0 texts...


Batches: 0it [00:00, ?it/s]

Generated embadings with shape: (0,)
Adding 0 documents to vector store...
error adding documents to vector store; Expected Embeddings to be non-empty list or numpy array, got [] in add.


ValueError: Expected Embeddings to be non-empty list or numpy array, got [] in add.

Retriver Pipeline From VectorStore

In [ ]:
class RAGRetriver:
    """Handle query-based retriveal from the vector store"""
    
    def __init__(self,vector_store: VectorSrore, embadding_manager: embadding_manager):
        """
        Initialize the retriver
        
        Args:
             Vector_store: Vectore store eontaining document embading
             embading_manager: Manager for generating query embading
        """
        self.vector_store = vector_store
        self.embadding_manager = embadding_manager
        
    def retrieve(self,query: str,top_k: int,score_threshold: float = 0.0) -> List[Dict[str, Any]]:  
        """
        Retrieve relevant document for query
        
        Args: 
             query: The search quary
             top_k: number of top result to return
             score_threshold: Minimum similarity score threshold
             
        Returns:
                List  of dictionaries containinf retrieved documents and metadata   
        """ 
        print(f"Retriveing douments for query: '{query}'")
        print(f"top_k: '{top_k}', Score threshold: {score_threshold}")
        
        #Generate query embading
        query_embedding = self.embadding_manager.generate_embaddings([query])[0]
        
        #Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            #Process result
            retrieved_docs = []
            
        
            if results['documents'] and results['documents'][0]:    
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances =  results['distances'][0]
                ids = results['ids'][0]
                
                
                for i, (doc_id,document,metadata,distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # convert distence to similarity score (ChromaDB uses cosine distances)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i+1
                        })
                print(f"retrived {len(retrieved_docs)} documents (after filtering)")
            
            else:
                print(f"No documents found")    
        
            return retrieved_docs  
        except Exception as e:
            print(f"Error during retrieval: {e}")
                           
                    
rag_retriver = RAGRetriver(
    vector_store=vectorstore,
    embadding_manager=Manager
)            
        

In [ ]:
rag_retriver.retrieve("tell me aboute just play",top_k=5)

Retriveing douments for query: 'tell me aboute just play'
top_k: '5', Score threshold: 0.0
Generating embaddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embadings with shape: (1, 384)
retrived 4 documents (after filtering)


[{'id': 'doc_c9a40f2c_8',
  'content': 'JustPlay',
  'metadata': {'content_length': 8,
   'source_file': 'Just Play final.pdf',
   'producer': 'macOS Version 13.3.1 (Build 22E261) Quartz PDFContext',
   'source': 'data\\pdf\\Just Play final.pdf',
   'title': 'Just Play',
   'total_pages': 58,
   'doc_index': 8,
   'file_type': 'pdf',
   'author': 'VRAJ PATEL',
   'creationdate': "D:20230519072542Z00'00'",
   'creator': 'Word',
   'page_label': '7',
   'page': 6,
   'moddate': "D:20230519072542Z00'00'"},
  'similarity_score': 0.3225822448730469,
  'distance': 0.6774177551269531,
  'rank': 1},
 {'id': 'doc_4440aab8_8',
  'content': 'JustPlay',
  'metadata': {'page_label': '7',
   'total_pages': 58,
   'page': 6,
   'source': 'data\\pdf\\Just Play final.pdf',
   'moddate': "D:20230519072542Z00'00'",
   'author': 'VRAJ PATEL',
   'content_length': 8,
   'source_file': 'Just Play final.pdf',
   'producer': 'macOS Version 13.3.1 (Build 22E261) Quartz PDFContext',
   'title': 'Just Play',
   

In [ ]:
answer = rag_simple("tell me aboute just play?", rag_retriver,llm)
print(answer)

NameError: name 'rag_simple' is not defined

Embadding Using Ollama Model

In [ ]:
# from langchain_community.vectorstores import FAISS
# import ollama
# from langchain_ollama import OllamaEmbeddings

# embaddings= OllamaEmbeddings(model="embeddinggemma-2:270m")
# vectorSet=FAISS.from_documents(chunks,embaddings)


# # response = ollama.embed(
# #     model="embeddinggemma-2:270m",
# #     input=chunks
# # )

In [ ]:
# documents_dir=DirectoryLoader(
#     'data/pdf',
#     glob='**/*.pdf',
#     loader_cls=PyPDFLoader,
#     show_progress=True
# )
# document=documents_dir.load()
# print(document)
# text_splitter=RecursiveCharacterTextSplitter(
#     chunk_size=1000,
#     chunk_overlap=200
# )
# splits=text_splitter.split_documents(document)
# splits[0]